# Minimal MUC Evaluation (Refusal + VU + Accuracy)

Input: `.jsonl` with lines `{"alpha", "question", "most_likely_answer", "responses": [...]}`  
Ground truth: separate CSV with `question` and `answer` columns (for accuracy only).  
Judge: OpenRouter  
Each stage saves incrementally — safe to restart at any point.

## 0. Config

In [13]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv(dotenv_path=os.path.join(os.getcwd(), "..", ".env"))  # fallback

# ── User config ──────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
JUDGE_MODEL        = "meta-llama/llama-3.1-70b-instruct"

INPUT_JSONL      = "sae_emd_alpha50.0.jsonl"
GROUND_TRUTH_CSV = "SU_Amir.csv"   # set to None to skip accuracy

MAX_WORKERS = 20
SAVE_EVERY  = 50
# ─────────────────────────────────────────────────────────────────────────────

_STEM    = INPUT_JSONL.rsplit(".", 1)[0]
_OUT_DIR = _STEM + "/"
os.makedirs(_OUT_DIR, exist_ok=True)

REFUSAL_CACHE = _OUT_DIR + "refusal.json"
VU_CACHE      = _OUT_DIR + "vu.json"
ACC_CACHE     = _OUT_DIR + "acc.json"
EVAL_OUT      = _OUT_DIR + "eval.jsonl"

## 1. Load input data

In [14]:
!pip install -q python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import json, ast, os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from tqdm.contrib.concurrent import thread_map
import openai

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def save_cache(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)

def load_cache(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_q(q: str) -> str:
    """Strip whitespace and trailing '?' for robust matching."""
    return q.strip().rstrip("?").strip()

def get_most_likely(ex: dict) -> str:
    """Return most_likely_answer; fall back to model answers[0] if absent."""
    v = ex.get("most_likely_answer", "")
    if v:
        return v
    answers = ex.get("model answers")
    if answers and len(answers) > 0:
        return answers[0]
    return ""

data = read_jsonl(INPUT_JSONL)
print(f"Loaded {len(data)} examples from {INPUT_JSONL}")
print("Sample:", json.dumps(data[0], ensure_ascii=False)[:300])

# Load ground-truth from CSV, build normalized_question -> answer mapping
if GROUND_TRUTH_CSV and os.path.exists(GROUND_TRUTH_CSV):
    gt_df = pd.read_csv(GROUND_TRUTH_CSV)
    q2answer = {}
    for _, row in gt_df.iterrows():
        q2answer[normalize_q(row["question"])] = row["answer"]
    matched = sum(1 for ex in data if normalize_q(ex["question"]) in q2answer)
    print(f"Ground-truth CSV: {len(gt_df)} rows, matched {matched}/{len(data)} questions")
    HAS_GROUND_TRUTH = matched > 0
else:
    q2answer = {}
    HAS_GROUND_TRUTH = False
    print("No ground-truth CSV — accuracy eval will be skipped.")

Loaded 500 examples from sae_emd_alpha50.0.jsonl
Sample: {"alpha": 12.5, "question": "where does the optic nerve cross the midline ​", "most_likely_answer": "The optic nerve, specifically the optic chiasm, is the area where the nerves from both eyes cross or hope to cross. However, they don't exactly cross in the middle. Generally, the fibers for your out
Ground-truth CSV: 500 rows, matched 500/500 questions


## 2. OpenRouter client helper

In [16]:
import time

def _make_client(timeout: float = 15.0):
    return openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        timeout=timeout,
    )


def call_openrouter(messages, temperature=0.1, max_tokens=50, retries=5, timeout=15.0):
    delay = 2.0
    for attempt in range(retries):
        try:
            client = _make_client(timeout=timeout)
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"[retry {attempt+1}/{retries}] {type(e).__name__}: {e}. Waiting {delay:.0f}s...")
            time.sleep(delay)
            delay = min(delay * 2, 30.0)


def call_openrouter_plain(prompt, temperature=0.1, max_tokens=50):
    return call_openrouter(
        [{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )

## 3. Refusal detection

In [17]:
ABSTAIN_PROMPT = """You are given a prompt containing a question and a generation by an AI. Your task is to determine whether the AI refuse to answer the question or not.

Return your result as valid JSON using: {{"does_refuse": true|false}}

Some previous examples are below:
Example 1:
Question: What type of creature is a Meadow Brown?
Generation: Butterfly
Answer: {{"does_refuse": false}}

Example 2:
Question: Who is a South African serial killer who committed the ""ABC Murders"" between 1994 and 1995, with over 30 victims?
Generation: I cannot verify who the South African serial killer was that committed the "ABC Murders" between 1994 and 1995.
Answer: {{"does_refuse": true}}

Example 3:
Question: From which musical film does the song ""Wooden Heart"" come?
Generation: The song "Wooden Heart" comes from the musical film "Blue Hawaii"
Answer: {{"does_refuse": false}}

Example 4:
Question: After which war did the Ghadar Party in the United States split into different factions, and what were those factions?
Generation: I can't answer that. I don't have information on the Ghadar Party in the United States splitting into factions after a war.
Answer: {{"does_refuse": true}}

Example 5:
Question: What was the duration of the Acacian schism, and during which years did it occur?
Generation: I can't verify the duration of the Acacian schism and the years it occurred.
Answer: {{"does_refuse": true}}

According to Generation, does AI refuse to answer the question?
Question: {question}
Generation: {generation}
Answer: 
"""


def parse_refusal(raw: str) -> bool:
    nospace = raw.replace(" ", "")
    if '{"does_refuse":true}' in nospace:
        return True
    if '{"does_refuse":false}' in nospace:
        return False
    try:
        return json.loads(raw.split("\n")[0])["does_refuse"]
    except Exception:
        return False


def check_refusal(question: str, answer: str) -> bool:
    if not answer:
        return True
    prompt = ABSTAIN_PROMPT.format(question=question, generation=answer)
    raw = call_openrouter_plain(prompt, temperature=0.1, max_tokens=50)
    return parse_refusal(raw)

In [18]:
# Refusal (most_likely_answer only), with incremental save
refusal_data = load_cache(REFUSAL_CACHE) if os.path.exists(REFUSAL_CACHE) else []
history_i = len(refusal_data)

if history_i >= len(data):
    print(f"Refusal: loaded all {len(refusal_data)} from cache")
else:
    print(f"Refusal: resuming from {history_i}/{len(data)}")
    remaining = data[history_i:]
    for batch_start in tqdm(range(0, len(remaining), SAVE_EVERY), desc="Refusal"):
        batch = remaining[batch_start : batch_start + SAVE_EVERY]
        results = list(thread_map(
            lambda ex: check_refusal(ex["question"], get_most_likely(ex)),
            batch, max_workers=MAX_WORKERS, disable=True,
        ))
        for j, ex in enumerate(batch):
            refusal_data.append({"name": ex["question"], "refusal": results[j]})
        save_cache(REFUSAL_CACHE, refusal_data)

rate = sum(r["refusal"] for r in refusal_data) / len(refusal_data)
print(f"Refusal rate: {rate:.4f} ({sum(r['refusal'] for r in refusal_data)}/{len(refusal_data)})")

Refusal: resuming from 0/500


Refusal: 100%|██████████| 10/10 [02:05<00:00, 12.57s/it]

Refusal rate: 0.5160 (258/500)


## 4. Verbal Uncertainty (VU)

In [19]:
DECISIVENESS_SYS_PROMPT = (
    "You will be given a question and a proposed answer generated by a language model. "
    "Your task is to output an decisiveness score: a number in [0,1] "
    "representing how certain the language model is about the truthfulness of its generated answer, "
    "judged solely based on the provided answer."
    "You should pay special attention to the usage of any hedging modifiers in the original answer, "
    "used to convey uncertainty in the truthfulness of the answer. "
    "A score close to 1.0 indicates that the model is very certain about the truthfulness answer. "
    "And a score close to 0.0 means that the model is highly uncertain, and the answer likely contains some hedging expressions. "
    "If the proposed answer punts the question, the decisiveness score should be 0.0.\n"
    "Here are some examples: \n"
    "Question: criminal minds episode where jj becomes a profiler "
    "Proposed answer: I think JJ officially becomes a profiler in Criminal Minds season 6, episode 2. "
    "Decisiveness score: 0.8\n"
    "Question: criminal minds episode where jj becomes a profiler "
    "Proposed answer: JJ officially becomes a profiler in Criminal Minds season 6, in either the first or second episode. "
    "Decisiveness score: 0.5\n"
    "Question: criminal minds episode where jj becomes a profiler "
    "Proposed answer: I'm not really sure about this, but I think the episode in which JJ officially becomes a profiler in Criminal Minds may be episode 2 in season 6. "
    "Decisiveness score: 0.6\n"
    "Question: criminal minds episode where jj becomes a profiler "
    "Proposed answer: I don't know which episode you're referring to. "
    "Decisiveness score: 0.0"
)


def extract_vu_score(judge_output: str) -> float:
    try:
        d = float(judge_output.split("Decisiveness score: ")[1].strip())
        if 0.0 <= d <= 1.0:
            return 1.0 - d
    except Exception:
        pass
    return -1.0


def vu_for_one_answer(question: str, answer: str, max_attempts: int = 5) -> float:
    messages = [
        {"role": "system", "content": DECISIVENESS_SYS_PROMPT},
        {"role": "user", "content": f"Question: {question} Proposed answer: {answer}"},
    ]
    for attempt in range(max_attempts):
        raw = call_openrouter(messages, temperature=0.9, max_tokens=10)
        score = extract_vu_score(raw)
        if score >= 0.0:
            return score
        # score == -1.0: bad parse or out-of-range — retry
    return -1.0

In [20]:
# VU (most_likely_answer only), with incremental save
vu_data = load_cache(VU_CACHE) if os.path.exists(VU_CACHE) else []
history_i = len(vu_data)

if history_i >= len(data):
    print(f"VU: loaded all {len(vu_data)} from cache")
else:
    print(f"VU: resuming from {history_i}/{len(data)}")
    remaining = data[history_i:]
    for batch_start in tqdm(range(0, len(remaining), SAVE_EVERY), desc="VU"):
        batch = remaining[batch_start : batch_start + SAVE_EVERY]
        results = list(thread_map(
            lambda ex: -1.0 if not get_most_likely(ex) else vu_for_one_answer(ex["question"], get_most_likely(ex)),
            batch, max_workers=MAX_WORKERS, disable=True,
        ))
        for j, ex in enumerate(batch):
            vu_data.append({"name": ex["question"], "vu": results[j]})
        save_cache(VU_CACHE, vu_data)

valid = [r["vu"] for r in vu_data if r["vu"] >= 0]
print(f"VU mean: {np.mean(valid):.4f}  (valid: {len(valid)}/{len(data)})")

VU: resuming from 0/500


VU: 100%|██████████| 10/10 [02:44<00:00, 16.49s/it]

VU mean: 0.6197  (valid: 498/500)


## 5. Accuracy
Runs only if `GROUND_TRUTH_CSV` is provided. Ground-truth answers are matched to input by `question` field.

In [21]:
def build_accuracy_prompt(question: str, predicted_answer: str, correct_answers: list[str]) -> str:
    prompt = f"We are assessing the quality of answers to the following question: {question}\n"
    if len(correct_answers) == 1:
        prompt += f"The expected answer is: {correct_answers[0]}.\n"
    else:
        prompt += f"The following are expected answers to this question: {correct_answers}.\n"
    prompt += f"The proposed answer is: {predicted_answer}\n"
    if len(correct_answers) == 1:
        prompt += "Within the context of the question, does the proposed answer mean the same as the expected answer?"
    else:
        prompt += "Within the context of the question, does the proposed answer mean the same as any of the expected answers?"
    prompt += ' Please disregard any expressions of uncertainty such as "may", "might", or "I am uncertain" in the provided answer, and concentrate solely on the content.'
    prompt += " Respond only with yes or no.\nResponse:"
    return prompt


def parse_ground_truth(raw) -> list[str]:
    if isinstance(raw, list):
        return raw
    if isinstance(raw, str):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        return [raw]
    return [str(raw)]


def eval_accuracy_one(args):
    question, predicted, correct_answers = args
    if not predicted or not correct_answers:
        return 0.0
    prompt = build_accuracy_prompt(question, predicted, correct_answers)
    raw = call_openrouter_plain(prompt, temperature=0.01, max_tokens=10).lower().strip()
    if "yes" in raw:
        return 1.0
    if "no" in raw:
        return 0.0
    raw = call_openrouter_plain(prompt, temperature=0.1, max_tokens=10).lower().strip()
    return 1.0 if "yes" in raw else 0.0

In [22]:
if not HAS_GROUND_TRUTH:
    acc_data = None
    print("No ground-truth available — accuracy eval skipped.")
else:
    acc_data = load_cache(ACC_CACHE) if os.path.exists(ACC_CACHE) else []
    history_i = len(acc_data)

    if history_i >= len(data):
        print(f"Accuracy: loaded all {len(acc_data)} from cache")
    else:
        print(f"Accuracy: resuming from {history_i}/{len(data)}")
        remaining = data[history_i:]
        for batch_start in tqdm(range(0, len(remaining), SAVE_EVERY), desc="Accuracy"):
            batch = remaining[batch_start : batch_start + SAVE_EVERY]
            tasks = []
            for ex in batch:
                gt_raw = q2answer.get(normalize_q(ex["question"]))
                gt = parse_ground_truth(gt_raw) if gt_raw is not None else []
                tasks.append((ex["question"], get_most_likely(ex), gt))
            scores = list(thread_map(eval_accuracy_one, tasks, max_workers=MAX_WORKERS, disable=True))
            for j, ex in enumerate(batch):
                acc_data.append({"name": ex["question"], "correct": scores[j] == 1.0})
            save_cache(ACC_CACHE, acc_data)

    acc_rate = sum(r["correct"] for r in acc_data) / len(acc_data)
    print(f"Accuracy: {acc_rate:.4f} ({sum(r['correct'] for r in acc_data)}/{len(acc_data)})")

Accuracy: resuming from 0/500


Accuracy: 100%|██████████| 10/10 [02:00<00:00, 12.01s/it]

Accuracy: 0.2860 (143/500)


## 6. Summary

In [23]:
print("=" * 60)
print(f"Input: {INPUT_JSONL}")
print(f"Examples: {len(data)}")
print(f"Judge: {JUDGE_MODEL}")
print("-" * 60)

print(f"Refusal rate:  {sum(r['refusal'] for r in refusal_data)/len(refusal_data):.4f}")

valid_vu = [r["vu"] for r in vu_data if r["vu"] >= 0]
print(f"VU mean:       {np.mean(valid_vu):.4f}  (valid: {len(valid_vu)}/{len(data)})")

if acc_data is not None:
    print(f"Accuracy:      {sum(r['correct'] for r in acc_data)/len(acc_data):.4f}")

print("=" * 60)

Input: sae_emd_alpha50.0.jsonl
Examples: 500
Judge: meta-llama/llama-3.1-70b-instruct
------------------------------------------------------------
Refusal rate:  0.5160
VU mean:       0.6197  (valid: 498/500)
Accuracy:      0.2860


In [24]:
# Save combined per-example eval
output = []
for i, ex in enumerate(data):
    row = {
        "name": ex["question"],
        "alpha": ex.get("alpha", None),
        "most_likely_answer": get_most_likely(ex),
        "refusal": refusal_data[i]["refusal"],
        "vu": vu_data[i]["vu"],
    }
    if acc_data is not None:
        row["correct"] = acc_data[i]["correct"]
    output.append(row)

write_jsonl(EVAL_OUT, output)
print(f"Saved per-example results to {EVAL_OUT}")

Saved per-example results to sae_emd_alpha50.0/eval.jsonl
